In [1]:
print("Hello World")

Hello World


In [2]:
!pip install tensorflow pandas scikit-learn

In [3]:
import tensorflow as tf

In [4]:
from tensorflow.keras.applications import DenseNet121

In [5]:
model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

In [6]:
model.trainable = False

In [7]:
model.summary

<bound method Model.summary of <Functional name=densenet121, built=True>>

In [8]:
from tensorflow.keras.layers import GlobalAveragePooling2D,Dense

In [12]:
x = GlobalAveragePooling2D()(model.output)
output = Dense(5,activation="softmax")(x)

In [13]:
MODEL = tf.keras.Model(inputs=model.input,outputs=output)

In [16]:
MODEL.summary

<bound method Model.summary of <Functional name=functional, built=True>>

In [19]:
MODEL.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=['accuracy']
)

In [21]:
MODEL.layers

[<InputLayer name=input_layer, built=True>,
 <ZeroPadding2D name=zero_padding2d, built=True>,
 <Conv2D name=conv1_conv, built=True>,
 <BatchNormalization name=conv1_bn, built=True>,
 <Activation name=conv1_relu, built=True>,
 <ZeroPadding2D name=zero_padding2d_1, built=True>,
 <MaxPooling2D name=pool1, built=True>,
 <BatchNormalization name=conv2_block1_0_bn, built=True>,
 <Activation name=conv2_block1_0_relu, built=True>,
 <Conv2D name=conv2_block1_1_conv, built=True>,
 <BatchNormalization name=conv2_block1_1_bn, built=True>,
 <Activation name=conv2_block1_1_relu, built=True>,
 <Conv2D name=conv2_block1_2_conv, built=True>,
 <Concatenate name=conv2_block1_concat, built=True>,
 <BatchNormalization name=conv2_block2_0_bn, built=True>,
 <Activation name=conv2_block2_0_relu, built=True>,
 <Conv2D name=conv2_block2_1_conv, built=True>,
 <BatchNormalization name=conv2_block2_1_bn, built=True>,
 <Activation name=conv2_block2_1_relu, built=True>,
 <Conv2D name=conv2_block2_2_conv, built=True>

In [22]:
from tensorflow.keras.applications.densenet import preprocess_input

image_size=(32,32)
batch_size=32

In [ ]:
train = tf.keras.utils.image_dataset_from_directory(
    './data/lungs-disease-dataset-4-types/versions/1/Lung Disease Dataset/train',
    image_size=image_size,
    batch_size=batch_size,
    label_mode="categorical",
    shuffle=True,
    seed=42
)

Found 6054 files belonging to 5 classes.


In [ ]:
valid = tf.keras.utils.image_dataset_from_directory(
    './data/lungs-disease-dataset-4-types/versions/1/Lung Disease Dataset/val',
    image_size=image_size,
    batch_size=batch_size,
    shuffle=False,
    label_mode="categorical"
)

Found 2016 files belonging to 5 classes.


In [33]:
test = tf.keras.utils.image_dataset_from_directory(
    './data/lungs-disease-dataset-4-types/versions/1/Lung Disease Dataset/test',
    image_size=image_size,
    batch_size=batch_size,
    shuffle=False,
    label_mode="categorical"
)

Found 2025 files belonging to 5 classes.


In [28]:
print("Classes:", train.class_names)

Classes: ['Bacterial Pneumonia', 'Corona Virus Disease', 'Normal', 'Tuberculosis', 'Viral Pneumonia']


In [29]:
AUTOTUNE = tf.data.AUTOTUNE
def preprocess(image,label):
    images = tf.cast(image, tf.float32)
    images = preprocess_input(images)
    return images, label

In [31]:
train = train.map(preprocess,num_parallel_calls=AUTOTUNE)
valid = valid.map(preprocess,num_parallel_calls=AUTOTUNE)

In [32]:
train = train.prefetch(AUTOTUNE)
valid = valid.prefetch(AUTOTUNE)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-7
    )
]

Learning has slowed down. Take smaller steps. - ReduceLROnPlateau

In [ ]:
history = MODEL.fit(
    train,
    validation_data=valid,
    callbacks=callbacks,
    verbose=1,
    epochs=100
)